In [ ]:
import nibabel as nib
import numpy as np
import pandas as pd


In [ ]:

# Load the GIFTI label file
fpath = "brain_rendering_code/my_schaefer_tian_files/Schaefer200_Tian_S2.L.32k_fs_LR.label.gii"
img = nib.load(fpath)

# --- Metadata ---
print("=== File Metadata ===")
for key, val in img.meta.metadata.items():
    print(f"  {key}: {val[:80] if len(val) > 80 else val}")


In [ ]:
# --- Label table ---
print("\n=== Label Table ===")
label_table = img.labeltable.labels  # list of GiftiLabel objects
rows = []
for label in label_table:
    rows.append({
        "key": label.key,
        "name": label.label,
        "rgba": (round(label.red,3), round(label.green,3), round(label.blue,3), round(label.alpha,3))
    })
df_labels = pd.DataFrame(rows)
print(f"Total labels: {len(df_labels)}")
print(df_labels.to_string())


In [ ]:

# --- Data array ---
print("\n=== Data Array ===")
data = img.darrays[0].data
print(f"Shape: {data.shape}  (vertices on 32k LH surface)")
print(f"dtype: {data.dtype}")
print(f"Unique parcel keys assigned: {np.unique(data)}")
print(f"Number of non-zero (labelled) vertices: {np.sum(data > 0)}")
print(f"Number of unlabelled (key=0) vertices: {np.sum(data == 0)}")

# --- Vertices per parcel ---
print("\n=== Vertex counts per parcel ===")
keys, counts = np.unique(data, return_counts=True)
df_counts = pd.DataFrame({"key": keys, "vertex_count": counts})
df_counts = df_counts.merge(df_labels[["key","name"]], on="key", how="left")
print(df_counts.to_string(index=False))

# --- Quick summary by region type ---
subcortical = df_labels[df_labels["key"].between(1, 32)]
cortical_lh = df_labels[df_labels["name"].str.startswith("7Networks_LH")]
cortical_rh = df_labels[df_labels["name"].str.startswith("7Networks_RH")]
print(f"\nSubcortical labels (Tian S2):  {len(subcortical)}")
print(f"Cortical LH parcels (Schaefer): {len(cortical_lh)}")
print(f"Cortical RH parcels (Schaefer): {len(cortical_rh)}")
